# Agent: modellen styrer søk og kalkulator

I agenten er det **modellen** som styrer. Den får spørsmålet og to verktøy, og bestemmer selv hva den gjør, steg for steg:

```
spørsmål → modell → verktøykall → observasjon → modell → … → svar
```

Dette er *agentløkken* fra forelesning 6: modell → verktøy → observasjon → modell. Modellen kjører aldri koden selv. Den *ber om* et verktøykall, Python utfører det, og resultatet sendes tilbake som en observasjon.

| Verktøy | Hva modellen kan gjøre | Felles kode |
|---|---|---|
| `search(query)` | Søke i spørsmålets egen rapportside, så mange ganger den vil, med egne formuleringer | `src/korpus.py` (samme søk som RAG-pipelinen, top_k = 5) |
| `calculator(op, a, b)` | Regne ett steg om gangen: add, subtract, multiply, divide | `src/kalkulator.py` (`calculator_tool`) |

**Sammenligning med RAG-pipelinen:**

| | RAG-pipeline (fast kjede) | Agent |
|---|---|---|
| Hvem bestemmer stegene? | Python: alltid retrieve → pakk → generer → regn | Modellen |
| Antall søk | Ett, med spørsmålet som query | Så mange modellen vil, innenfor maks 8 steg |
| Regning | Ett helt program i én runde | Ett steg om gangen, og den ser mellomresultatene |
| Modellkall | 1 | 1–8 |
| Retry og selvkorrigering | Nei | Ja: modellen kan prøve på nytt etter en feil |

**Alt annet er likt** og hentes fra `src/`: modell, korpus, søk, kalkulator, tallregler i prompten, utvalg, korrekthetsregel og loggformat.

## 0. Oppsett

Alle innstillingene ligger i `src/config.py`, og det er den samme filen RAG-pipelinen bruker. Her importerer vi dem og viser dem. Det eneste som velges i denne notebooken, er **hvilket utvalg** vi kjører på.

**Lagrede resultater.** Notebooken gjør ingen API-kall. Kjøringen på hele testutvalget leses fra `results/`, og det er disse resultatene som står i rapporten. **Valgfritt ekstra steg:** seksjon 9 sist i notebooken kan kjøre agenten live mot OpenAI på 8 testspørsmål, hvis du vil se metoden i bruk med ekte API-kall (krever egen API-nøkkel).

**Demo (ca. 1–2 minutter, noen cent).** Kjør alle cellene. Med `LIVE_DEMO = True` kjøres agenten live mot OpenAI på de `N_DEMO` = 8 første testspørsmålene (seksjon 5). Svarene sammenlignes med den lagrede testkjøringen. Alt etter demoen, med evaluering på alle 200 testspørsmålene, leser den lagrede testkjøringen i `results/`. Uten API-nøkkel i `.env` hoppes live-cellene over, og resten av notebooken kjører som vanlig.

In [1]:
# --- Les inn endringer i src/ automatisk ---
# Uten dette husker Python den gamle versjonen av en modul selv om filen er endret,
# og nye funksjoner (som classify_failure) blir "not defined" til kjernen startes på nytt.
%load_ext autoreload
%autoreload 2

import json        # verktøyargumenter kommer som JSON-tekst fra modellen
import re          # regulære uttrykk: finner "ANSWER: ..." i modellens svar
import sys         # gir tilgang til Pythons søkesti, så vi kan importere fra src/
import time        # stoppeklokke: tidsbruk per spørsmål er en del av kostnadsmålet
from pathlib import Path

import numpy as np
import pandas as pd

# --- Finn prosjektroten og gjør src/ importerbar ---
# Notebooken ligger i notebooks/, så prosjektroten er ett nivå opp.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

# --- Felles innstillinger (samme fil som RAG-pipelinen bruker) ---
# MODEL og TEMPERATURE: samme språkmodell som RAG-pipelinen (gpt-4.1, temperatur 0).
# TOP_K: antall chunks per søk, likt for RAG-pipelinen og hvert av agentens søk.
# MAX_AGENT_STEPS: stoppkriteriet i agentløkken (forel. 6), maks antall modellkall per spørsmål.
from src.config import MODEL, TEMPERATURE, TOP_K, MAX_AGENT_STEPS, RESULTS_DIR, settings

# --- Felles byggeklosser ---
# Alt under er den samme koden som RAG-pipelinen importerer. Det eneste agenten lager selv i denne
# notebooken, er verktøyskjemaene, systemprompten og agentløkken.
from src.utvalg import load_utvalg, load_examples          # samme spørsmål som RAG-pipelinen
from src.korpus import Korpus, gold_coverage               # samme korpus og søk som RAG-pipelinen
from src.kalkulator import calculator_tool, reset_log, get_log   # samme kalkulator som RAG-pipelinen
from src.prompts import NUMBER_RULES                       # samme tallregler som RAG-pipeline-prompten
from src.llm import call_model, usage                      # samme modellkall og tokentelling
from src.evaluering import (parse_answer, is_correct, is_correct_rounded, is_scale_error,
                            format_answer, classify_failure)      # samme retting som RAG-pipelinen
from src.logg import make_record, write_jsonl, read_jsonl, same_settings  # samme loggfelt som RAG-pipelinen
from src.rapport import oppsummering, failure_log, failure_oversikt   # samme evalueringstabeller som RAG-pipelinen

# --- Det eneste valget i denne notebooken: hvilket utvalg ---
# "dev": all utvikling og justering skjer her.
# "test": kjøres ÉN gang, når både RAG-pipelinen og agenten er ferdige. Ikke bytt før det!
SPLIT = "test"
CONDITION = "Agent"    # betingelsen, står i hver loggrad

pd.set_option("display.max_colwidth", 120)

# --- Vis innstillingene ---
# settings() er det samme som logges i hver rad, så vi alltid vet hvilket oppsett et resultat kom fra.
{"condition": CONDITION, "split": SPLIT, **settings()}

{'condition': 'Agent',
 'split': 'test',
 'model': 'gpt-4.1',
 'temperature': 0,
 'seed': 42,
 'retrieval_scope': 'dokument',
 'chunk_strategy': 'setning',
 'index_backend': 'embedding',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'top_k': 5,
 'max_agent_steps': 8,
 'rel_tol': 0.01}

## 1. Utvalg og korpus

Agenten bruker **nøyaktig de samme spørsmålene og det samme korpuset** som RAG-pipelinen:

- `load_utvalg(SPLIT)` leser `data/utvalg_dev.csv`, den samme filen RAG-pipelinen leser.
- `Korpus(...)` chunker hver rapportside og lager embeddinger for alle chunkene **én gang**. Det er samme kode som RAG-pipelinen bruker (`src/korpus.py`).

Til slutt prøver vi søket på ett spørsmål. Treffet med spørsmålet selv som query er **nøyaktig det RAG-pipelinen får**. Agenten kan i tillegg søke med egne formuleringer, og det er der den kan hente evidens som RAG-pipelinen går glipp av.

`is_gold` viser om chunken inneholder evidens fasiten bygger på (`gold_inds`), som i RAG-pipelinen.

In [2]:
# --- Steg 1: Les utvalget (samme fil som RAG-pipelinen) ---
# utvalg: én rad per spørsmål, med spørsmål, fasit (exe_ans), fasitprogram og gold_inds.
# examples: oppslag fra spørsmålets id til hele rapportsiden (pre_text, tabell, post_text).
utvalg = load_utvalg(SPLIT)
examples = load_examples(SPLIT)     # id -> hele FinQA-eksempelet (tekst, tabell, qa)

# --- Steg 2: Bygg korpuset én gang, bare for spørsmålene i utvalget ---
# Indekseringstid (forel. 4): hver rapportside deles i chunks, og hver chunk får en embedding.
# Dette skjer ÉN gang her. Senere, i spørretid, søker agenten i det ferdige korpuset.
start = time.time()
korpus = Korpus({qid: examples[qid] for qid in utvalg["id"]})
print({"split": SPLIT, "spørsmål": len(utvalg),
       "chunks_totalt": sum(len(c) for c in korpus.chunks.values()),
       "backend": korpus.backend, "sekunder": round(time.time() - start, 1)})

# Fordeling av regnesteg i fasitprogrammet (avtalt inndeling: ett vs. flere steg)
display(utvalg["n_steps"].map(lambda n: "1 steg" if n == 1 else "2+ steg")
        .value_counts().rename("antall spørsmål").to_frame())

# --- Steg 3: Prøv søket på ett spørsmål ---
# Med spørsmålet som query er dette nøyaktig det RAG-pipelinen henter.
# is_gold = True betyr at chunken er en av dem fasiten bygger på (gold_inds).
eksempel = utvalg.iloc[0]
hits = pd.DataFrame(korpus.search(eksempel["question"], doc_id=eksempel["id"]))
hits["is_gold"] = hits["source_ids"].map(lambda ids: bool(set(ids) & set(eksempel["gold_inds"])))
print({"spørsmål": eksempel["question"], "gold_inds": eksempel["gold_inds"], "fasit": float(eksempel["gold_answer"])})
hits[["rank", "score", "chunk_id", "kind", "is_gold", "text"]]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'split': 'test', 'spørsmål': 200, 'chunks_totalt': 5601, 'backend': 'embedding', 'sekunder': 12.6}


,antall spørsmål
n_steps,
1 steg,110
2+ steg,90


{'spørsmål': 'what is the average number of shares per registered holder as of february 17 , 2017?', 'gold_inds': ['text_3'], 'fasit': 2792124.42484}


,rank,score,chunk_id,kind,is_gold,text
0,1,0.705,text_3,tekst,True,"as of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered holders ."
1,2,0.566,text_1,tekst,False,"market for registrant 2019s common equity , related stockholder matters and issuer purchases of equity securities th..."
2,3,0.524,text_2,tekst,False,"on february 17 , 2017 , the closing price of our common stock was $ 108.11 per share as reported on the nyse ."
3,4,0.509,text_9,tekst,False,"we have distributed an aggregate of approximately $ 3.2 billion to our common stockholders , including the dividend ..."
4,5,0.486,text_6,tekst,False,"we have two series of preferred stock outstanding , 5.25% ( 5.25 % ) mandatory convertible preferred stock , series ..."


## 2. Verktøyene

Hvert verktøy har **to deler**:

1. **Skjemaet** er det modellen ser: navn, beskrivelse og hvilke argumenter verktøyet tar (JSON Schema). Beskrivelsen er det eneste modellen vet om verktøyet, så den avgjør når og hvordan det blir brukt (forel. 6).
2. **Python-funksjonen** kjøres når modellen ber om verktøyet. Modellen kjører aldri noe selv.

| Verktøy | Modellen sender | Python gjør | Modellen får tilbake |
|---|---|---|---|
| `search` | `{"query": "..."}` | `korpus.search(query, doc_id)`, med top_k = 5 i spørsmålets eget dokument | De 5 chunkene i **samme format som RAG-pipeline-konteksten**: `[chunk-ID \| type]` + tekst |
| `calculator` | `{"op": "subtract", "a": 290, "b": 92}` | `calculator_tool(op, a, b)` fra `src/kalkulator.py` | Resultatet (`198.0`), eller `FEIL: ...` |

**Hvorfor `doc_id` ikke er et argument:** modellen skal bare bestemme *hva* den søker etter. *Hvor* den søker, i spørsmålets egen rapportside, bestemmes av Python, akkurat som i RAG-pipelinen. Derfor lager vi en ny `ToolBox` for hvert spørsmål, med `doc_id` bundet på forhånd.

**Feil krasjer ikke agenten.** Ukjent verktøy, ugyldige argumenter eller deling på null gir en tekst som starter med `FEIL:`. Modellen kan lese den og prøve igjen. Det er en del av det som skiller agenten fra RAG-pipelinen.

**Alt logges:** hvert kall (navn, argumenter, output) og hver chunk som ble hentet. Det blir `tool_calls` og `retrieved_chunks` i loggen.

In [3]:
# --- Del 1: Skjemaene, det modellen ser ---
# Forelesning 6: modellen kjenner verktøyene BARE gjennom skjemaet (navn, beskrivelse, argumenter).
# Beskrivelsen er derfor en del av prompten: den avgjør når og hvordan modellen bruker verktøyet.
# Formatet er Responses API sitt: type, name, description, parameters (JSON Schema).
# strict=True: modellen MÅ sende nøyaktig disse argumentene, ingen andre.

# Søkeverktøyet: modellen bestemmer bare query (HVA den leter etter).
# HVOR den leter (doc_id) er ikke et argument. Det bindes i ToolBox under, som i RAG-pipelinen.
SEARCH_SCHEMA = {
    "type": "function",
    "name": "search",
    "description": ("Search the financial report for this question. Returns the 5 most relevant "
                    "sentences or table rows, each with an id like [text_3] or [table_2]. "
                    "You can search several times with different wording, e.g. a row label, "
                    "a year or a term from the question."),
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "What to look for in the report."}},
        "required": ["query"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Kalkulatorverktøyet: ett regnesteg om gangen.
# "enum" begrenser op til de fire regneartene kalkulatoren kan, og "number" gjør at a og b
# kommer som tall, ikke som tekst med tusenskille eller valutategn.
CALCULATOR_SCHEMA = {
    "type": "function",
    "name": "calculator",
    "description": ("Compute ONE arithmetic step: op(a, b). Use it for every calculation, "
                    "and use earlier results as numbers in later steps."),
    "parameters": {
        "type": "object",
        "properties": {
            "op": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
            "a": {"type": "number", "description": "First number (no thousands separators)."},
            "b": {"type": "number", "description": "Second number (no thousands separators)."},
        },
        "required": ["op", "a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Listen som sendes med i hvert modellkall i agentløkken.
TOOLS = [SEARCH_SCHEMA, CALCULATOR_SCHEMA]


# --- Hjelpefunksjon: treff som tekst, i SAMME format som RAG-pipeline-konteksten ---
# Hvert treff blir "[chunk-ID | type]" + tekst, og treffene skilles med en tom linje.
# Da ser agenten og RAG-pipelinen evidensen presentert likt, og forskjellen ligger bare i styringen.
def format_hits(hits: list) -> str:
    return "\n\n".join(f"[{h['chunk_id']} | {h['kind']}]\n{h['text']}" for h in hits)


# --- Del 2: Python-funksjonene, det som faktisk kjøres ---
class ToolBox:
    """Verktøyene for ETT spørsmål. doc_id er bundet, så modellen bare styrer query."""

    def __init__(self, doc_id: str):
        # Steg 1: Bind verktøyene til spørsmålets egen rapportside (RETRIEVAL_SCOPE = "dokument").
        self.doc_id = doc_id
        # Steg 2: Tomme logger. De fylles mens agenten jobber, og blir til
        # tool_calls og retrieved_chunks i resultatfilen.
        self.calls = []          # alle verktøykall: {"name", "arguments", "output"}
        self.retrieved = {}      # alle hentede chunks: chunk_id -> {"text", "source_ids"}

    def search(self, query: str) -> str:
        # Samme søkefunksjon som RAG-pipelinen, men med modellens egen query i stedet for spørsmålet.
        hits = korpus.search(query, doc_id=self.doc_id)            # samme søk som RAG-pipelinen, top_k = TOP_K
        # Husk hver chunk som er hentet, i ALLE søkene. setdefault: en chunk som hentes
        # flere ganger, lagres bare én gang. Dette brukes til retrieval-dekningen.
        for h in hits:
            self.retrieved.setdefault(h["chunk_id"], {"text": h["text"], "source_ids": h["source_ids"]})
        # Modellen får treffene tilbake som tekst, det er observasjonen i agentløkken.
        return format_hits(hits)

    def calculator(self, op: str, a: float, b: float) -> str:
        # Den felles kalkulatoren kaster aldri feil til agenten: ved feil (f.eks. deling på null)
        # får modellen teksten "FEIL: ..." og kan prøve igjen. Det kan ikke RAG-pipelinen.
        return str(calculator_tool(op, a, b))   # felles kalkulator: tall, eller "FEIL: ..."

    def run(self, name: str, arguments: str) -> str:
        """Kjør verktøyet modellen ba om. arguments er JSON-tekst fra modellen."""
        # Steg 1: Les argumentene. Ugyldig JSON gir en feilmelding, ikke et krasj.
        args = arguments                         # logges som rå tekst hvis JSON-en er ugyldig
        try:
            args = json.loads(arguments)
        except json.JSONDecodeError as exc:
            output = f"FEIL: ugyldige argumenter ({exc})"
        else:
            # Steg 2: Kjør riktig funksjon. **args pakker ut {"query": ...} til search(query=...).
            if name == "search":
                output = self.search(**args)
            elif name == "calculator":
                output = self.calculator(**args)
            else:
                output = f"FEIL: ukjent verktøy {name!r}"
        # Steg 3: Logg kallet (navn, argumenter og hva modellen fikk tilbake).
        self.calls.append({"name": name, "arguments": args, "output": output})
        return output


# --- Prøv verktøyene for hånd, slik modellen ville brukt dem ---
# Ingen modellkall her: vi later som vi er modellen og sender argumentene som JSON-tekst.
tools = ToolBox(eksempel["id"])
print(tools.run("search", json.dumps({"query": "capital expenditures retail segment incurred during 2003"})))
print("\n---")
print("calculator ->", tools.run("calculator", json.dumps({"op": "subtract", "a": 290, "b": 92})))
# En bevisst feil: viser at agenten får en feilmelding tilbake i stedet for at programmet krasjer.
print("feil       ->", tools.run("calculator", json.dumps({"op": "divide", "a": 1, "b": 0})))

# Loggen over kallene, som blir tool_calls i resultatfilen
pd.DataFrame(tools.calls)

[table_6 | tabell]
quarter ended march 31: high = $ 101.88; low = $ 93.21

[table_1 | tabell]
quarter ended march 31: high = $ 102.93; low = $ 83.07

[text_9 | tekst]
we have distributed an aggregate of approximately $ 3.2 billion to our common stockholders , including the dividend paid in january 2017 , primarily subject to taxation as ordinary income. .

[table_8 | tabell]
quarter ended september 30: high = 101.54; low = 86.83

[table_9 | tabell]
quarter ended december 31: high = 104.12; low = 87.23

---
calculator -> 198.0
feil       -> FEIL: deling på null (divide(1, 0))


,name,arguments,output
0,search,{'query': 'capital expenditures retail segment incurred during 2003'},[table_6 | tabell]\nquarter ended march 31: high = $ 101.88; low = $ 93.21\n\n[table_1 | tabell]\nquarter ended marc...
1,calculator,"{'op': 'subtract', 'a': 290, 'b': 92}",198.0
2,calculator,"{'op': 'divide', 'a': 1, 'b': 0}","FEIL: deling på null (divide(1, 0))"


## 3. Systemprompten

Systemprompten er instruksen modellen får før spørsmålet. Den gjør tre ting:

1. **Forklarer arbeidsmåten:** søk etter tallene, søk på nytt med andre ord hvis noe mangler, og bruk kalkulatoren til all regning. Agenten skal *ikke* regne i hodet. Da ville vi målt modellens hoderegning og ikke verktøybruken.
2. **Gjenbruker tallreglene fra RAG-pipelinen** (`NUMBER_RULES` fra `src/prompts.py`), som avtalt. Begge systemene får nøyaktig samme instruks om tusenskille, minustegn og andel i stedet for prosent.
3. **Bestemmer svarformatet:** siste linje skal være `ANSWER: <tall>`. Det gjør svaret lett å finne i teksten. Hvis modellen glemmer formatet, tar `parse_answer` det siste tallet i teksten.

**Modellen skal alltid gi et svar**, også når den er usikker. RAG-pipelinen gir alltid et program. Hvis agenten fikk lov til å avstå, ville de to systemene blitt målt på ulike premisser.

**Grensen på 8 steg** står i prompten, slik at modellen vet at den ikke kan søke i det uendelige. Selve grensen håndheves i Python, i agentløkken i celle 4.

In [4]:
# --- Systemprompten ---
# Arbeidsmåte og svarformat er agentens egne. Tallreglene er de samme som i RAG-pipelinen (NUMBER_RULES).
# f-strengen setter inn NUMBER_RULES og MAX_AGENT_STEPS, så prompten alltid følger src/.
#   - Punkt 1 ber om nye søk med andre ord: det er agentens fordel over RAG-pipelinens ene søk.
#   - Punkt 2 krever kalkulatoren til all regning, så vi måler verktøybruk og ikke hoderegning.
#   - Punkt 3 og "Always give a final answer" gjør at agenten, som RAG-pipelinen, aldri kan avstå.
AGENT_SYSTEM_PROMPT = f"""You answer numerical questions about a financial report.
You have two tools:
- search: finds sentences and table rows in the report.
- calculator: computes one arithmetic step, op(a, b).

Work like this:
1. Search the report for the numbers you need. If a search does not give you all the numbers,
   search again with different words, for example a row label, a year or a term from the question.
2. Use the calculator for EVERY arithmetic step. Do not calculate in your head.
3. When you have the result, give your final answer. The last line must be:
   ANSWER: <number>

Numbers:
{NUMBER_RULES}
- The final answer is one number in decimal form.

Always give a final answer, even if you are unsure. You have at most {MAX_AGENT_STEPS} turns."""


# --- Hent det endelige svaret ut av modellens tekst ---
# Først linjen "ANSWER: ...". Finnes den ikke, tar parse_answer det siste tallet i teksten.
# parse_answer normaliserer prosent: "14%" -> 0.14 (felles regel i src/evaluering.py).
# re.IGNORECASE: "Answer:" og "answer:" godtas også.
ANSWER_RE = re.compile(r"ANSWER:\s*(.+)", re.IGNORECASE)

def extract_final_answer(text: str) -> float | None:
    # Steg 1: Finn alle "ANSWER:"-linjer. text or "" tåler at modellen ikke svarte (None).
    match = ANSWER_RE.findall(text or "")
    # Steg 2: Bruk den SISTE ANSWER-linjen, eller hele teksten hvis det ikke finnes noen.
    return parse_answer(match[-1] if match else text)


# --- Vis prompten og test svarhentingen ---
# Fem typiske svar: riktig format, prosent, uten ANSWER-linje, negativt tall og ikke noe tall.
print(AGENT_SYSTEM_PROMPT)
print("\n--- Test av extract_final_answer ---")
for tekst in ["The change is 198 million.\nANSWER: 198",
              "ANSWER: 1.7%",
              "The ratio is about 0.0171",
              "ANSWER: -0.024",
              "I could not find the numbers."]:
    print(f"{tekst!r:45} -> {extract_final_answer(tekst)}")

You answer numerical questions about a financial report.
You have two tools:
- search: finds sentences and table rows in the report.
- calculator: computes one arithmetic step, op(a, b).

Work like this:
1. Search the report for the numbers you need. If a search does not give you all the numbers,
   search again with different words, for example a row label, a year or a term from the question.
2. Use the calculator for EVERY arithmetic step. Do not calculate in your head.
3. When you have the result, give your final answer. The last line must be:
   ANSWER: <number>

Numbers:
- Write numbers without thousands separators and without currency signs: 1234.5, not $1,234.5
- Write negative numbers with a minus sign: -32, not (32)
- Percentages and ratios are answered as fractions in decimal form. Do NOT multiply by 100.
  A change from 50 to 60 is 0.2, not 20. 12.4% is written 0.124.
- Use only numbers that appear in the report.
- The final answer is one number in decimal form.

Always give

## 4. Agentløkken

Dette er hjertet i agenten, og den eneste delen som virkelig skiller seg fra RAG-pipelinen. Løkken følger forelesning 6:

```
┌──────────────────────────────────────────────────────────────┐
│  1. Send samtalen så langt + verktøyene til modellen          │
│  2. Ba modellen om verktøy?                                   │
│        ja  → Python kjører verktøyet (ToolBox.run)           │
│              → resultatet legges til samtalen (observasjon)   │
│              → tilbake til 1                                  │
│        nei → modellen har svart → ferdig                      │
│  Siste steg (MAX_AGENT_STEPS): verktøy slås av → tvunget svar │
└──────────────────────────────────────────────────────────────┘
```

**Samtalen vokser for hvert steg.** Alt modellen har gjort (søk, resultater, regnesteg) sendes med i neste kall. Det er slik modellen «husker» hva den har funnet, og det er grunnen til at agenten bruker mange flere tokens enn RAG-pipelinen.

**Hvert modellsvar kan inneholde flere verktøykall** samtidig, for eksempel to søk. Alle kjøres før neste modellkall.

**Grensen på 8 modellkall** håndheves her. På **siste steg** får modellen beskjed om å svare nå, og verktøyene slås av (`tool_choice="none"`). Da gir agenten alltid et svar, på samme måte som RAG-pipelinen alltid gir et program, og den kan ikke «avstå» ved å søke til tiden er ute. Grensen på 8 modellkall holder fortsatt, fordi det tvungne svaret *er* det 8. kallet.

Dette ble lagt til etter testen på dev: uten det søkte agenten 12 ganger på ett spørsmål, brukte 14 400 tokens og ga aldri noe svar. `stopped` i loggen viser om svaret var frivillig (`svar`) eller tvunget (`tvunget_svar`).

**Sporet** (`trace`) logger hvert steg: hva modellen ba om, med hvilke argumenter, og hva den fikk tilbake. Det er dette som gjør agenten inspiserbar, på samme måte som retrieval-loggen i RAG-pipelinen.

In [5]:
# Beskjeden modellen får på siste steg, når verktøyene slås av.
FINAL_STEP_MESSAGE = ("You have reached the step limit. Do not search or calculate any more. "
                      "Give your best final answer now. The last line must be: ANSWER: <number>")


def agent_loop(question: str, doc_id: str, max_steps: int = MAX_AGENT_STEPS) -> dict:
    # --- Steg 0: Klargjør for ett spørsmål ---
    # Nye verktøy for hvert spørsmål, bundet til spørsmålets dokument, med tomme logger.
    tools = ToolBox(doc_id)                                 # verktøy bundet til spørsmålets dokument
    # Samtalen starter med bare spørsmålet. Systemprompten sendes for seg (instructions=...).
    conversation = [{"role": "user", "content": f"Question: {question}"}]
    # trace: sporet steg for steg (hva modellen gjorde og fikk tilbake), til visning og feilsøking.
    # tokens: summeres over ALLE modellkallene, fordi hele samtalen sendes på nytt hver gang.
    trace, tokens = [], {"input_tokens": 0, "output_tokens": 0, "reasoning_tokens": 0}
    # Standardverdier: hvis løkken går ut uten svar, står stopped igjen som "maks_steg".
    final_text, stopped = None, "maks_steg"

    # --- Agentløkken (forel. 6): modell -> verktøykall -> observasjon -> modell -> ... ---
    # Hver runde er ETT modellkall. step teller fra 1 til max_steps (8).
    for step in range(1, max_steps + 1):
        # Siste steg: be om svaret nå og slå av verktøyene, så agenten alltid svarer.
        last_step = step == max_steps
        if last_step:
            conversation.append({"role": "user", "content": FINAL_STEP_MESSAGE})

        # Steg 1: Ett modellkall med hele samtalen så langt og verktøyene.
        # tool_choice="none" på siste steg: modellen ser verktøyene, men kan ikke kalle dem.
        # Ellers (None) bestemmer modellen selv om den vil bruke et verktøy eller svare.
        response = call_model(conversation, instructions=AGENT_SYSTEM_PROMPT, tools=TOOLS,
                              tool_choice="none" if last_step else None)
        # Legg tokenbruken for dette kallet til summen (kostnadsmålet).
        for key, value in usage(response).items():
            tokens[key] += value

        # Modellens output (inkludert verktøykall og eventuelle resonneringssteg) legges i samtalen,
        # slik at modellen ser hva den selv gjorde i forrige runde.
        # Modellen har ingen hukommelse mellom kall: samtalelisten ER hukommelsen.
        conversation += [item.model_dump(exclude_none=True) for item in response.output]

        # Steg 2: Ba modellen om verktøy?
        # Svaret kan inneholde tekst, ett eller flere verktøykall ("function_call"), eller begge.
        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            # Nei: modellen har gitt et svar. Ferdig.
            # "tvunget_svar" hvis det skjedde på siste steg (verktøyene var slått av), ellers "svar".
            final_text, stopped = response.output_text, "tvunget_svar" if last_step else "svar"
            trace.append({"steg": step, "type": stopped, "verktøy": "", "argumenter": "", "resultat": final_text})
            break

        # Steg 3: Ja: kjør hvert verktøykall og legg resultatet (observasjonen) i samtalen.
        # Python utfører verktøyet, ikke modellen. call_id kobler resultatet til riktig kall,
        # slik at modellen vet hvilket søk eller regnesteg resultatet hører til.
        for call in function_calls:
            output = tools.run(call.name, call.arguments)
            conversation.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
            trace.append({"steg": step, "type": "verktøykall", "verktøy": call.name,
                          "argumenter": call.arguments, "resultat": output})
        # Tilbake til toppen av løkken: neste modellkall ser nå observasjonene.

    # --- Steg 4: Samle alt fra kjøringen ---
    return {
        "final_text": final_text,                       # råteksten fra modellen (None hvis maks steg)
        "answer": extract_final_answer(final_text),     # parset tall
        "stopped": stopped,                             # "svar", "tvunget_svar" eller "maks_steg"
        "n_model_calls": step,                          # løkken stoppet på dette steget
        "trace": trace,
        "tools": tools,                                 # tool_calls og retrieved_chunks
        **tokens,
    }


# --- Sporet for eksempelspørsmålet fra den lagrede testkjøringen ---
# Løkken kjøres ikke her (den gjør API-kall). Verktøykallene agenten gjorde på dette
# spørsmålet i testkjøringen, er lagret i results/. En valgfri live-kjøring finnes i seksjon 9.
lagret_eksempel = read_jsonl(RESULTS_DIR / f"agent_{SPLIT}.jsonl").set_index("id").loc[eksempel["id"]]
print({"spørsmål": eksempel["question"], "fasit": float(eksempel["gold_answer"]),
       "svar": float(lagret_eksempel["answer"]), "riktig": bool(lagret_eksempel["correct"]),
       "stoppet": lagret_eksempel["stopped"], "modellkall": int(lagret_eksempel["n_model_calls"]),
       "verktøykall": int(lagret_eksempel["n_tool_calls"])})
spor = pd.DataFrame(lagret_eksempel["tool_calls"])
spor["output"] = spor["output"].astype(str).str.slice(0, 150)     # søkeresultatene er lange: vis starten
spor

{'spørsmål': 'what is the average number of shares per registered holder as of february 17 , 2017?', 'fasit': 2792124.42484, 'svar': 2792124.424836601, 'riktig': True, 'stoppet': 'svar', 'modellkall': 3, 'verktøykall': 3}


,name,arguments,output
0,search,"{'query': 'number of shares as of February 17, 2017'}","[text_3 | tekst]\nas of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered ..."
1,search,"{'query': 'number of registered holders as of February 17, 2017'}","[text_3 | tekst]\nas of february 17 , 2017 , we had 427195037 outstanding shares of common stock and 153 registered ..."
2,calculator,"{'op': 'divide', 'a': 427195037, 'b': 153}",2792124.424836601


## 5. Én loggrad per spørsmål: `run_agent()`

`run_agent` kjører agentløkken og gjør resultatet om til **én loggrad med de avtalte feltene** (`src/logg.py`). RAG-pipelinen lager nøyaktig de samme feltene, så de to resultatfilene kan legges side om side og kobles på `id`.

| Felt | Agenten | RAG-pipeline (til sammenligning) |
|---|---|---|
| `answer` / `raw_answer` | Tallet fra `ANSWER:`-linjen / hele sluttsvaret | Kalkulatorens resultat / programmet |
| `tool_calls` | Alle `search`- og `calculator`-kall | Ett `run_program`-kall |
| `retrieved_chunks` | Alle chunks fra **alle** søk | De 5 chunkene fra ett søk |
| `n_model_calls` | 1–8 | Alltid 1 |
| `stopped` | `svar` eller `tvunget_svar` (siste steg) | Alltid `svar` |
| `gold_support_retrieved` | Ble alle `gold_inds` hentet i **minst ett** av søkene? | Ble alle hentet i det ene søket? |

**Retting og feilsted** bruker de felles funksjonene i `src/evaluering.py`:
- `is_correct`: relativ feil < 1 %.
- `classify_failure`: samme rekkefølge som RAG-pipelinen. Agenten har i tillegg `maks_steg`, som betyr at den ikke rakk å svare. For agenten betyr `kalkulator` at sluttsvaret ikke kunne leses som et tall.

**API-feil** (nettverk, kvote) fanges og lagres i `error`. De er ikke agentfeil og holdes utenfor evalueringen.

Funksjonen kjøres ikke her. Resultatene for hele testutvalget leses inn i seksjon 6. En valgfri live-kjøring på 8 testspørsmål finnes i seksjon 9.

In [6]:
def run_agent(q: pd.Series) -> dict:
    # --- Steg 0: Klargjør ---
    start = time.time()
    reset_log()                                   # tøm kalkulatorens interne logg før hvert spørsmål
    gold = float(q["gold_answer"])
    # Feltene som er kjent før agenten starter: hvilket spørsmål, fasit og oppsett.
    # De kommer med i loggraden uansett, også om kjøringen feiler.
    base = dict(condition=CONDITION, split=SPLIT, id=q["id"], question=q["question"], gold_answer=gold,
                gold_program=q["gold_program"], gold_inds=list(q["gold_inds"]), n_steps=int(q["n_steps"]),
                model=MODEL, settings=settings())

    # Steg 1: Kjør agentløkken. API-feil er ikke agentfeil: de lagres i error.
    try:
        run = agent_loop(q["question"], q["id"])
    except Exception as exc:
        return make_record(**base, error=f"{type(exc).__name__}: {exc}", seconds=round(time.time() - start, 2))

    # Steg 2: Retrieval-dekning over ALLE søkene agenten gjorde.
    # RAG-pipelinen måles på sitt ene søk. Agenten måles på summen av alt den hentet, fordi det er
    # nettopp muligheten til å søke flere ganger vi undersøker.
    retrieved = [{"chunk_id": cid, **info} for cid, info in run["tools"].retrieved.items()]
    all_gold, any_gold = gold_coverage(retrieved, q["gold_inds"])

    # Steg 3: Retting og feilsted med de felles reglene.
    # is_correct: relativ feil < 1 % mot exe_ans (hovedregelen, lik for RAG-pipelinen).
    correct = is_correct(run["answer"], gold)
    # Steg 4: Én loggrad med de avtalte feltene (src/logg.py), de samme som RAG-pipelinen.
    return make_record(
        **base,
        # Svaret: det parsede tallet og hele sluttsvaret fra modellen.
        answer=run["answer"], raw_answer=run["final_text"],
        # Verktøybruken og alt som ble hentet (grunnlaget for failure-loggen).
        tool_calls=run["tools"].calls,
        retrieved_chunks=[{"chunk_id": r["chunk_id"], "source_ids": r["source_ids"], "text": r["text"]} for r in retrieved],
        # Kostnad: modellkall, verktøykall, tokens og tid.
        n_model_calls=run["n_model_calls"], n_tool_calls=len(run["tools"].calls), stopped=run["stopped"],
        input_tokens=run["input_tokens"], output_tokens=run["output_tokens"],
        reasoning_tokens=run["reasoning_tokens"],
        seconds=round(time.time() - start, 2), error=None,
        # Retting: hovedregel, tilleggsmål (3 desimaler) og retrieval-dekning.
        correct=correct, correct_rounded=is_correct_rounded(run["answer"], gold),
        gold_support_retrieved=all_gold, any_gold_retrieved=any_gold,
        # Feilsted (forel. 4): hvilket ledd som feilet, i fast rekkefølge (src/evaluering.py).
        failure_stage=classify_failure(correct, run["answer"], gold, all_gold, stopped=run["stopped"]),
    )

## 6. Kjøring på hele utvalget

Nå kjører vi `run_agent` på alle spørsmålene i `SPLIT`, som er dev (50 spørsmål) under utviklingen og test (200 spørsmål) i den endelige kjøringen.

**Lagring:** `results/agent_{SPLIT}.jsonl`, én JSON-linje per spørsmål med de avtalte feltene. RAG-pipelinen lagrer til `results/rag_pipeline_{SPLIT}.jsonl` med de samme feltene, så filene kan kobles på `id`.

**Ingen dobbeltbetaling:** finnes filen, leses den inn i stedet for å kjøre på nytt. Sett `RERUN = True` når du har endret noe i agenten, for eksempel prompten.

**API-feil** (`error` er satt) er ikke agentfeil. De vises for seg, og kan kjøres på nytt.

Agenten bruker flere modellkall enn RAG-pipelinen, så dette tar noen minutter. Framdriften skrives ut for hvert tiende spørsmål.

In [7]:
RERUN = False                                         # True = kjør alle spørsmålene på nytt, også med uendret oppsett
RESULTS_PATH = RESULTS_DIR / f"agent_{SPLIT}.jsonl"        # f.eks. results/agent_dev.jsonl

# --- Steg 0: Gjenbruk en lagret kjøring bare hvis den ble laget med samme oppsett som nå ---
# Er f.eks. modellen byttet i src/config.py, er filen utdatert, og alt kjøres på nytt automatisk.
# same_settings sammenligner settings-feltet i de lagrede radene med settings() nå.
lagret = read_jsonl(RESULTS_PATH) if RESULTS_PATH.exists() else None
if lagret is not None and not RERUN and not same_settings(lagret, settings()):
    print(f"{RESULTS_PATH.relative_to(ROOT)} ble laget med et annet oppsett enn src/config.py, kjører på nytt")

if lagret is not None and not RERUN and same_settings(lagret, settings()):
    # --- Steg 1a: Les inn tidligere kjøring ---
    resultater = lagret
    print(f"Leste {len(resultater)} resultater fra {RESULTS_PATH.relative_to(ROOT)} (sett RERUN = True for å kjøre på nytt)")
else:
    # --- Steg 1b: Kjør agenten på hvert spørsmål ---
    # Ett spørsmål om gangen. Hvert run_agent-kall gir én loggrad (1-8 modellkall).
    rader, start = [], time.time()
    for i, (_, q) in enumerate(utvalg.iterrows(), start=1):
        rader.append(run_agent(q))
        # Framdrift for hvert tiende spørsmål, så vi ser at kjøringen går.
        if i % 10 == 0:
            print(f"{i}/{len(utvalg)} ferdig, {time.time() - start:.0f} s")

    # --- Steg 2: Lagre som JSONL (avtalt format) ---
    # Én JSON-linje per spørsmål, med de samme feltene som RAG-pipelinens fil, så de kan kobles på id.
    write_jsonl(rader, RESULTS_PATH)
    resultater = pd.DataFrame(rader)
    print(f"Lagret {len(resultater)} resultater til {RESULTS_PATH.relative_to(ROOT)}")

# --- Steg 3: Skill ut API-feil (ikke agentfeil) ---
# ok er radene som evalueres videre. API-feil vises for seg, så de kan kjøres på nytt.
api_feil = resultater[resultater["error"].notna()]
ok = resultater[resultater["error"].isna()]
if len(api_feil):
    display(api_feil[["id", "error"]])

# --- Steg 4: Rask oversikt (grundig evaluering kommer i celle 7) ---
print({"spørsmål": len(resultater), "api_feil": len(api_feil),
       "riktige": int(ok["correct"].sum()),
       "tvungne_svar": int((ok["stopped"] == "tvunget_svar").sum()),
       "modellkall_snitt": round(float(ok["n_model_calls"].mean()), 1),
       "tokens_totalt": int(ok["input_tokens"].sum() + ok["output_tokens"].sum())})
ok[["id", "gold_answer", "answer", "correct", "failure_stage", "stopped", "n_model_calls", "n_tool_calls"]].head(10)

Leste 200 resultater fra results/agent_test.jsonl (sett RERUN = True for å kjøre på nytt)
{'spørsmål': 200, 'api_feil': 0, 'riktige': 129, 'tvungne_svar': 31, 'modellkall_snitt': 4.7, 'tokens_totalt': 904158}


,id,gold_answer,answer,correct,failure_stage,stopped,n_model_calls,n_tool_calls
0,AMT/2016/page_49.pdf-4,2.792124e+06,2.792124e+06,True,ingen,svar,3.0,3.0
1,ETR/2011/page_301.pdf-3,7.416150e+04,1.158900e+04,False,uttrykk,tvunget_svar,8.0,7.0
2,CDW/2015/page_93.pdf-2,1.262070e+00,2.014000e+03,False,retrieval,svar,7.0,7.0
3,GS/2013/page_63.pdf-2,-2.704800e+04,-2.704800e+04,True,ingen,svar,3.0,2.0
4,JPM/2008/page_87.pdf-1,6.200000e-01,6.200000e-01,True,ingen,svar,3.0,2.0
5,FIS/2016/page_9.pdf-1,8.008500e-01,8.008475e-01,True,ingen,svar,4.0,3.0
6,FIS/2017/page_64.pdf-1,-4.138000e-02,-4.138292e-02,True,ingen,svar,5.0,4.0
7,SYY/2005/page_70.pdf-2,7.190300e-01,7.190000e-01,True,ingen,svar,5.0,4.0
8,INTC/2013/page_31.pdf-2,-2.200000e-01,-2.200000e-01,True,ingen,svar,7.0,6.0
9,RSG/2015/page_98.pdf-2,2.147750e+00,1.003000e+02,False,retrieval,tvunget_svar,8.0,8.0


## 7. Evaluering og failure-logg

**Felles mål.** Tabellene lages av de samme funksjonene som i RAG-pipelinen (`src/rapport.py`). Tallene kan derfor legges rett ved siden av hverandre:

1. Nøyaktighet med 95 % konfidensintervall (hovedregel 1 % og tilleggsmål 3 desimaler)
2. Retrieval recall mot `gold_inds`, over **alle** søkene agenten gjorde
3. Nøyaktighet med og uten evidens
4. Ett vs. flere regnesteg
5. Feilsteder
6. Kostnad per spørsmål

**Agentspesifikt: hvordan brukte agenten verktøyene?** Dette finnes ikke i RAG-pipelinen, fordi RAG-pipelinen alltid gjør det samme. Her ser vi om agenten faktisk utnytter friheten sin:
- **Søkte den mer enn én gang?** Det er der den kan hente evidens RAG-pipelinen går glipp av.
- **Brukte den kalkulatoren?** Prompten sier at den skal, men en modell kan regne i hodet likevel. Et riktig svar uten kalkulatorkall betyr at vi måler hoderegning, ikke verktøybruk.
- **Hvor ofte ble svaret tvunget** (8 steg nådd)?

**Failure-loggen** har samme form som i RAG-pipelinen, med den uavhengige testen `tall_i_kontekst`: stod fasittallene i noe agenten hentet? Den lagres som `results/agent_{SPLIT}_failure_log.csv`.

Med 50 spørsmål (dev) er intervallene omtrent ±13 prosentpoeng, med 200 (test) omtrent ±7. Dev er til for å justere, test for å konkludere.

In [8]:
# --- Del 1: De felles tabellene (samme som RAG-pipelinen) ---
# oppsummering (src/rapport.py) lager nøyaktighet med 95 % Wilson-KI, retrieval recall,
# nøyaktighet med/uten evidens, ett vs. flere regnesteg, feilsteder og kostnad.
tabeller = oppsummering(ok)
for navn in ["noyaktighet", "recall", "etter_evidens", "etter_steg", "feilsted", "kostnad"]:
    print(f"--- {navn} ---")
    display(tabeller[navn])

# --- Del 2: Hvordan brukte agenten verktøyene? ---
# Steg 1: Tell søk og kalkulatorkall per spørsmål ut fra tool_calls-loggen.
bruk = ok.copy()
bruk["søk"] = bruk["tool_calls"].map(lambda calls: sum(c["name"] == "search" for c in calls))
bruk["kalkulatorkall"] = bruk["tool_calls"].map(lambda calls: sum(c["name"] == "calculator" for c in calls))
# Steg 2: Oppsummer. Tre spørsmål vi vil svare på:
#   - Utnytter agenten friheten til å søke flere ganger? (andel med mer enn ett søk)
#   - Følger den prompten og bruker kalkulatoren? (andel uten kalkulatorkall = mulig hoderegning)
#   - Hvor ofte nådde den stegsgrensen og måtte svare? (andel tvungne svar)
print("--- verktøybruk ---")
display(pd.DataFrame([{
    "søk per spørsmål (snitt)": round(float(bruk["søk"].mean()), 1),
    "andel med mer enn ett søk": f"{(bruk['søk'] > 1).mean():.0%}",
    "kalkulatorkall per spørsmål (snitt)": round(float(bruk["kalkulatorkall"].mean()), 1),
    "andel uten kalkulatorkall": f"{(bruk['kalkulatorkall'] == 0).mean():.0%}",
    "andel tvungne svar": f"{(bruk['stopped'] == 'tvunget_svar').mean():.0%}",
}]).T.rename(columns={0: "Agent"}))

# Steg 3: Nøyaktighet etter hvor mange ganger agenten søkte: hjelper ekstra søk?
# pd.cut deler antall søk i tre grupper: 0-1, 2 og 3 eller flere.
# Merk: mange søk kan også bety at evidensen var vanskelig å finne, ikke at søk skader.
bruk["søkegrupper"] = pd.cut(bruk["søk"], bins=[-1, 1, 2, 100], labels=["1 søk", "2 søk", "3+ søk"])
print("--- nøyaktighet etter antall søk ---")
display(bruk.groupby("søkegrupper", observed=True).agg(
    antall=("id", "size"), riktige=("correct", "sum"),
    alle_gold_hentet=("gold_support_retrieved", lambda s: f"{s.mean():.0%}")))

# --- Del 3: Failure-logg (samme form som RAG-pipelinen) ---
# Én rad per feil svar, med feilsted og den uavhengige kontrollen tall_i_kontekst:
# stod alle tallene fra fasitprogrammet i noe agenten hentet? ("les ferdig kontekst", forel. 4)
logg = failure_log(ok)
print("--- feilsteder og uavhengig evidenstest ---")
display(failure_oversikt(logg, len(ok)))
logg.to_csv(RESULTS_DIR / f"agent_{SPLIT}_failure_log.csv", index=False)
logg[["failure_stage", "tall_i_kontekst", "question", "raw_answer", "gold_program", "answer", "gold_answer"]]

--- noyaktighet ---


,mål,andel [95 % KI],antall
0,"riktig, relativ feil < 1 % (hovedregel)","64% [58%, 71%]",129
1,"riktig, 3 desimaler (tilleggsmål)","64% [57%, 70%]",127


--- recall ---


,mål,andel [95 % KI]
0,alle gold_inds hentet,"76% [70%, 82%]"
1,minst én gold_ind hentet,"94% [90%, 97%]"


--- etter_evidens ---


,antall,riktige,andel
gold_support_retrieved,,,
gold mangler,47,15,32%
alle gold hentet,153,114,75%


... (9 outputs left out)

,antall,riktige,alle_gold_hentet
søkegrupper,,,
1 søk,100,79,87%
2 søk,52,38,81%
3+ søk,48,12,50%


--- feilsteder og uavhengig evidenstest ---


,antall,fasittall_i_kontekst,andel_av_alle_spm
failure_stage,,,
uttrykk,37,37,18%
retrieval,32,6,16%
formatering,2,2,1%


,failure_stage,tall_i_kontekst,question,raw_answer,gold_program,answer,gold_answer
43,formatering,True,what is the five year total return on the goldman sachs group inc.?,ANSWER: 1.4836,"subtract(248.36, const_100)",1.4836,148.36000
182,formatering,True,what was the difference in cumulative total stockholder return percentage for illumina inc . common stock versus the...,The difference in cumulative total stockholder return percentage for Illumina Inc. common stock versus the Nasdaq Ph...,"subtract(43.81, const_100), divide(#0, const_100), subtract(74.57, const_100), divide(#2, const_100), subtract(#0, #2)",-0.3076,-30.76000
2,retrieval,True,"how much more money was expensed per outstanding basic weighted-average share in the year ended dec 31 , 2013 compar...","I have not found the total expenses for the years ended December 31, 2013 and 2014, only the expenses for specific p...","divide(156.6, 17.3), divide(170.6, 21.9), subtract(#0, #1)",2014.0000,1.26207
9,retrieval,False,in december 2015 what was the ratio of the restricted cash and marketable securities to the allowance for doubtful a...,"I found that the restricted cash and marketable securities as of December 31, 2015, was 100.3 million. However, I wa...","divide(100.3, 46.7)",100.3000,2.14775
11,retrieval,False,what percent of the commitments are due after 2019?,The report does not provide a clear breakdown of commitments due after 2019 as a percentage of total commitments. Th...,"divide(613, 1178)",0.0000,0.52037
...,...,...,...,...,...,...,...
176,uttrykk,True,how bigger were the interest and penalties concerning the interest income in the year 2015?,ANSWER: 6,"divide(8, 2), subtract(#0, 1)",6.0000,3.00000
187,uttrykk,True,what was the increase in the settlements with tax authorities as a percent of the tax liabilities observed during 20...,The increase in settlements with tax authorities from 2013 to 2014 is -21. The sum of tax liabilities (using the end...,"add(29, 4), add(8, 2), divide(29, #0), divide(8, #1), subtract(#2, #3)",-0.2140,0.07879
190,uttrykk,True,"what are the total off-balance sheet obligations , ( in millions ) ?",ANSWER: 1171.0,"add(505, 165)",1171.0000,670.00000
194,uttrykk,True,what is the increase in operation and maintenance expenses as a percentage of net revenue in 2003?,"I could not find the exact total amount for operation and maintenance expenses in 2003, only the increases due to sp...","add(6.6, 3.7), divide(#0, 426.6)",0.0000,0.02414


## 8. Sammenligning med RAG-pipelinen (McNemar)

Begge systemene har svart på **de samme 200 testspørsmålene**, så de sammenlignes parvis (koblet på `id`). Tallene leses fra de lagrede filene `results/agent_test.jsonl` og `results/rag_pipeline_test.jsonl`. Ingen modeller kjøres på nytt.

**McNemars eksakte test**, den samme som i `pilot_val.ipynb`: spørsmål der begge har rett eller begge har feil, sier ingenting om hvilket system som er best. Testen ser bare på de **uenige** spørsmålene, der bare det ene systemet har rett. Er systemene like gode, er hvert uenige spørsmål et myntkast (p = 0,5), og p-verdien er sannsynligheten for en minst like skjev fordeling. Funksjonen er `mcnemar_sammenligning` i `src/rapport.py`.

Testen vises for hovedregelen (`correct`, relativ feil under 1 %) og tilleggsmålet (`correct_rounded`, likt etter avrunding til tre desimaler).

In [9]:
# --- Steg 1: Les de lagrede testkjøringene for begge systemene (ingen API-kall) ---
from src.rapport import mcnemar_sammenligning

agent_test = read_jsonl(RESULTS_DIR / "agent_test.jsonl")
rag_test = read_jsonl(RESULTS_DIR / "rag_pipeline_test.jsonl")

# --- Steg 2: Parvis sammenligning med eksakt McNemar-test, for begge korrekthetsreglene ---
# Radene kobles på id. A = agenten, B = RAG-pipelinen.
mcnemar = pd.DataFrame([
    mcnemar_sammenligning(agent_test, rag_test, kolonne=regel, navn_a="agenten", navn_b="RAG-pipelinen")
    for regel in ["correct", "correct_rounded"]
]).set_index("regel")
mcnemar["p (McNemar, eksakt)"] = mcnemar["p (McNemar, eksakt)"].round(3)
mcnemar.astype(object).T          # object: telletallene vises som heltall, ikke 200.000

regel,correct,correct_rounded
spørsmål,200,200
riktige (agenten),129,127
riktige (RAG-pipelinen),121,120
begge riktig,113,112
begge feil,63,65
bare agenten riktig,16,15
bare RAG-pipelinen riktig,8,8
"p (McNemar, eksakt)",0.152,0.21


## 9. Ekstra: live-demo mot API-et (valgfritt)

Alt over bruker den lagrede testkjøringen, som er grunnlaget for rapporten. Dette steget er et **supplement** for den som vil se agenten i bruk med ekte API-kall.

- Sett `LIVE_DEMO = True` i cellen under og kjør den. Det krever `OPENAI_API_KEY` i `.env`.
- Agenten kjøres på de `N_DEMO` = 8 første testspørsmålene, med de samme funksjonene som over.
- Live-svarene vises ved siden av svarene fra den lagrede testkjøringen. Med temperatur 0 blir de stort sett like, men små forskjeller kan forekomme.
- Demoen lagrer ingenting og endrer ikke resultatene over. Den tar under ett minutt og koster omtrent 10 cent.

In [9]:
# --- Valgfritt: sett til True for å kjøre live mot OpenAI (krever OPENAI_API_KEY i .env) ---
LIVE_DEMO = False

from src.config import N_DEMO                     # antall testspørsmål i demoen (8)
from src.llm import har_api_nokkel                # finnes det en API-nøkkel i .env?
from src.rapport import demo_sammenligning        # live-svar ved siden av den lagrede testkjøringen

if not LIVE_DEMO:
    print("Live-demoen er av (LIVE_DEMO = False). Alle resultatene over er fra den lagrede testkjøringen.")
elif not har_api_nokkel():
    print("Fant ingen OPENAI_API_KEY i .env. Legg inn en nøkkel for å kjøre live-demoen.")
else:
    # Steg 1: Kjør systemet live på de N_DEMO første spørsmålene i utvalget.
    demo = pd.DataFrame([run_agent(q) for _, q in utvalg.head(N_DEMO).iterrows()])
    # Tell hvor mange ganger agenten søkte og regnet på hvert spørsmål.
    demo["søk"] = demo["tool_calls"].map(lambda calls: sum(c["name"] == "search" for c in calls) if calls else 0)
    demo["regnesteg"] = demo["tool_calls"].map(lambda calls: sum(c["name"] == "calculator" for c in calls) if calls else 0)
    # Steg 2: Legg svarene fra den lagrede testkjøringen ved siden av (src/rapport.py).
    sammendrag, demo = demo_sammenligning(demo, RESULTS_DIR / f"agent_{SPLIT}.jsonl")
    print(sammendrag)
    display(demo[[c for c in ["id", "gold_answer", "answer", "correct", "svar_testkjøring", "samme_svar",
                              "failure_stage", "n_model_calls", "søk", "regnesteg", "input_tokens", "seconds"] if c in demo]])
    # Steg 3: Sporet for det første spørsmålet: hva agenten søkte etter og regnet ut, steg for steg.
    spor = pd.DataFrame(demo.iloc[0]["tool_calls"])
    spor["output"] = spor["output"].astype(str).str.slice(0, 150)
    print(f"--- Sporet for {demo.iloc[0]['id']} ---")
    display(spor)

Live-demoen er av (LIVE_DEMO = False). Alle resultatene over er fra den lagrede testkjøringen.
